# Smart Water Monitor — Entrenamiento limpio del LSTM Autoencoder

**Antes de ejecutar (checklist):**
1. Session options → Accelerator: **GPU T4 x2** · Internet: **On**
2. **Add Input** → tu dataset privado con `mixed_population_dataset_160_households_more_leaks.csv`
3. Ejecutar con **Save Version → Save & Run All (Commit)** — corre en batch hasta 12h aunque cierres el navegador.

**Metodología** (corrige las fugas de datos del notebook original):
- Split **por hogar** (75/25): el test son hogares que el modelo jamás ve.
- Scaler ajustado **solo con filas normales de hogares de train**.
- Umbral (percentil 96 de validación) guardado como artefacto `threshold.json` junto al modelo.

Objetivo a batir: **F1 diario 0.696 de la regla MNF** (evaluación 2026-07-05 sobre hogares no vistos).


In [ ]:
# === 1. Setup ===
# Usamos el TensorFlow PREINSTALADO de Kaggle: forzar otra version via pip
# rompe numpy (quedan mezclados numpy 1.x y 2.x -> RecursionError al importar).
# La compatibilidad con el backend local se garantiza guardando TAMBIEN los
# pesos en .h5 (formato tolerante entre versiones de Keras).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.7.2"],
               check=True)  # solo sklearn (pin del scaler); no toca numpy ni TF

import numpy as np, sklearn
import tensorflow as tf
print("TF:", tf.__version__, "| sklearn:", sklearn.__version__, "| numpy:", np.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "Sin GPU: revisa Session options -> Accelerator"


In [ ]:
# === 2. Localizar el dataset adjuntado ===
import glob
candidates = glob.glob("/kaggle/input/**/mixed_population_dataset_160_households_more_leaks.csv",
                       recursive=True)
assert candidates, "No encuentro el CSV: usa Add Input y adjunta tu dataset con el fichero more_leaks"
DATA_PATH = candidates[0]
print("Dataset:", DATA_PATH)


In [ ]:
# === 3. Feature engineering ===
# COPIA de backend/app/ml/features.py (misma implementacion que usa el detector
# en produccion). Si cambias algo aqui, cambia tambien el fichero del repo.
import numpy as np
import pandas as pd

ROLLING_WINDOWS = [4, 8, 12, 24, 48, 96]
LAGS = [96, 96 * 7]
SEQUENCE_LENGTH = 96


def engineer_features(data):
    df = data.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values(["household_id", "timestamp"]).reset_index(drop=True)

    for col, period in [("hour", 24), ("dayofweek", 7)]:
        df[f"{col}_sin"] = np.sin(2 * np.pi * getattr(df["timestamp"].dt, col) / period)
        df[f"{col}_cos"] = np.cos(2 * np.pi * getattr(df["timestamp"].dt, col) / period)

    df["is_weekend"] = (df["timestamp"].dt.dayofweek >= 5).astype(int)
    df["is_night"] = df["timestamp"].dt.hour.between(0, 5, inclusive="both").astype(int)

    grouped = df.groupby("household_id", sort=False)["consumption_l"]

    for window in ROLLING_WINDOWS:
        rolling = grouped.rolling(window=window, min_periods=1)
        rmean = rolling.mean().reset_index(level=0, drop=True)
        rstd = rolling.std().reset_index(level=0, drop=True)
        df[f"rolling_mean_{window}"] = rmean
        df[f"rolling_std_{window}"] = rstd
        df[f"rolling_cv_{window}"] = rstd / (rmean + 1e-6)
        df[f"diff_from_rolling_mean_{window}"] = df["consumption_l"] - rmean

    df.fillna(0, inplace=True)

    for lag in LAGS:
        df[f"consumption_lag_{lag}"] = grouped.shift(lag).fillna(0)

    feature_cols = [c for c in df.columns if c not in ["timestamp", "is_leak", "household_id"]]
    return df, feature_cols


def create_sequences(df, feature_cols, seq_length=SEQUENCE_LENGTH, stride=2):
    sequences, labels, metadata = [], [], []
    has_labels = "is_leak" in df.columns

    for household_id, group in df.groupby("household_id"):
        group = group.sort_values("timestamp")
        values = group[feature_cols].values
        leak_flags = group["is_leak"].values if has_labels else np.zeros(len(group))
        timestamps = group["timestamp"].values
        if len(group) < seq_length:
            continue
        num_seq = (len(group) - seq_length) // stride + 1
        starts = np.arange(num_seq) * stride
        for s in starts:
            e = s + seq_length
            sequences.append(values[s:e])
            labels.append(int(np.max(leak_flags[s:e])))
            metadata.append({"household_id": household_id, "end_timestamp": timestamps[e - 1]})

    return np.stack(sequences), np.array(labels, dtype=int), metadata

print("Features listas")


In [ ]:
# === 4. Configuracion y modelo (de backend/app/ml/train_clean.py) ===
from pathlib import Path
from tensorflow.keras import callbacks, layers, models, optimizers

CFG = dict(
    sequence_length=SEQUENCE_LENGTH,
    sequence_stride=4,          # stride 4: mitad de datos/RAM que el original (2)
    test_household_fraction=0.25,
    validation_split=0.15,
    split_seed=42,
    latent_dim=64,
    lstm_units=(128, 64),       # red compacta; si no bate al MNF, probar (256,128,64)
    batch_size=256,
    max_epochs=150,
    patience=15,
    threshold_percentile=96.0,
)

OUTPUT_DIR = Path("/kaggle/working/Run_Clean")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
RESULTS_DIR = OUTPUT_DIR / "results"
for p in (CHECKPOINT_DIR, RESULTS_DIR):
    p.mkdir(parents=True, exist_ok=True)


def split_households(household_ids, cfg):
    rng = np.random.RandomState(cfg["split_seed"])
    ids = sorted(household_ids)
    rng.shuffle(ids)
    n_test = max(1, int(len(ids) * cfg["test_household_fraction"]))
    return ids[n_test:], ids[:n_test]


def build_lstm_autoencoder(input_shape, latent_dim, lstm_units):
    inputs = layers.Input(shape=input_shape)
    x = inputs
    for units in lstm_units:
        x = layers.LSTM(units, return_sequences=True)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.2)(x)
    x = layers.LSTM(lstm_units[-1] // 2, return_sequences=False)(x)
    x = layers.BatchNormalization()(x)
    latent = layers.Dense(latent_dim, activation="relu", name="bottleneck")(x)
    x = layers.RepeatVector(input_shape[0])(latent)
    for units in reversed(lstm_units):
        x = layers.LSTM(units, return_sequences=True)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.2)(x)
    outputs = layers.TimeDistributed(layers.Dense(input_shape[1]))(x)
    model = models.Model(inputs, outputs, name="lstm_autoencoder")
    model.compile(optimizer=optimizers.Adam(1e-4), loss="mse")
    return model

print("Config lista:", CFG)


In [ ]:
# === 5. Pipeline SIN FUGA DE DATOS: split por hogar -> scaler solo train-normal ===
import joblib
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42)
np.random.seed(42)

print("Leyendo dataset...")
df = pd.read_csv(DATA_PATH)

all_ids = df["household_id"].unique().tolist()
train_ids, test_ids = split_households(all_ids, CFG)
print(f"Hogares: {len(train_ids)} train / {len(test_ids)} test (nunca vistos)")

df, feature_cols = engineer_features(df)

df_train = df[df["household_id"].isin(train_ids)].copy()
df_test = df[df["household_id"].isin(test_ids)].copy()
del df

scaler = StandardScaler()
scaler.fit(df_train.loc[df_train["is_leak"] == 0, feature_cols])
df_train[feature_cols] = scaler.transform(df_train[feature_cols]).astype(np.float32)
df_test[feature_cols] = scaler.transform(df_test[feature_cols]).astype(np.float32)
joblib.dump(scaler, RESULTS_DIR / "scaler.joblib")
print("Scaler guardado (fit solo con train-normal)")

X_train_all, y_train_all, _ = create_sequences(df_train, feature_cols,
                                               CFG["sequence_length"], CFG["sequence_stride"])
X_test, y_test, test_meta = create_sequences(df_test, feature_cols,
                                             CFG["sequence_length"], CFG["sequence_stride"])
del df_train, df_test

X_train_normal = X_train_all[y_train_all == 0]
del X_train_all, y_train_all
val_size = int(len(X_train_normal) * CFG["validation_split"])
X_val = X_train_normal[-val_size:]
X_train = X_train_normal[:-val_size]

print(f"Secuencias - train(normal): {len(X_train):,} | val: {len(X_val):,} | "
      f"test(mixto): {len(X_test):,} (fugas: {int(y_test.sum()):,})")


In [ ]:
# === 6. Entrenamiento (reanudable: BackupAndRestore) ===
model = build_lstm_autoencoder((CFG["sequence_length"], len(feature_cols)),
                               CFG["latent_dim"], list(CFG["lstm_units"]))
model.summary()

cbs = [
    callbacks.ModelCheckpoint(filepath=str(CHECKPOINT_DIR / "best_model.keras"),
                              monitor="val_loss", save_best_only=True, verbose=1),
    callbacks.CSVLogger(str(OUTPUT_DIR / "training_log.csv"), append=True),
    callbacks.EarlyStopping(monitor="val_loss", patience=CFG["patience"],
                            restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, verbose=1),
    callbacks.BackupAndRestore(backup_dir=str(OUTPUT_DIR / "backup_state")),
]

history = model.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=CFG["max_epochs"],
    batch_size=CFG["batch_size"],
    callbacks=cbs,
    shuffle=True,
    verbose=2,
)
print("Entrenamiento terminado")


In [ ]:
# === 7. Umbral + evaluacion honesta sobre hogares nunca vistos ===
import json
from datetime import datetime
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score)

best_model = models.load_model(CHECKPOINT_DIR / "best_model.keras")
# Pesos en h5: red de seguridad si el .keras no carga en el TF local (2.16.2)
best_model.save_weights(str(CHECKPOINT_DIR / "best_model.weights.h5"))

def reconstruction_errors(m, X, bs=512):
    return np.mean(np.square(X - m.predict(X, batch_size=bs, verbose=0)), axis=(1, 2))

val_errors = reconstruction_errors(best_model, X_val)
threshold = float(np.percentile(val_errors, CFG["threshold_percentile"]))
print(f"Umbral (p{CFG['threshold_percentile']} de validacion normal): {threshold:.6f}")

test_errors = reconstruction_errors(best_model, X_test)
y_pred = (test_errors >= threshold).astype(int)

prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary",
                                                   zero_division=0)
metrics = {
    "precision": float(prec),
    "recall": float(rec),
    "f1": float(f1),
    "roc_auc": float(roc_auc_score(y_test, test_errors)),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    "classification_report": classification_report(y_test, y_pred, output_dict=True,
                                                   zero_division=0),
}

artifact = {
    "threshold": threshold,
    "threshold_percentile": CFG["threshold_percentile"],
    "trained_at": datetime.now().isoformat(timespec="seconds"),
    "data_path": DATA_PATH,
    "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()},
    "train_households": sorted(train_ids),
    "test_households": sorted(test_ids),
    "methodology": "household-split, scaler fit on train-normal only (no leakage)",
    "metrics_on_unseen_households": metrics,
}
(RESULTS_DIR / "threshold.json").write_text(json.dumps(artifact, indent=2))

print("=" * 60)
print("METRICAS HONESTAS a nivel de SECUENCIA (hogares nunca vistos):")
print(f"   ROC-AUC:   {metrics['roc_auc']:.4f}")
print(f"   Recall:    {metrics['recall']:.3f}")
print(f"   Precision: {metrics['precision']:.3f}")
print(f"   F1:        {metrics['f1']:.3f}")
print("Recuerda: el listado a batir es el F1 DIARIO 0.696 del MNF ->")
print("evalua en local con: python -m app.ml.evaluate_ensemble tras desplegar artefactos")
print("=" * 60)


In [ ]:
# === 8. Empaquetar artefactos para descargar ===
import shutil
shutil.make_archive("/kaggle/working/Run_Clean_artifacts", "zip", str(OUTPUT_DIR))
print("Listo: descarga Run_Clean_artifacts.zip desde el panel Output de la version")


## Despliegue en tu backend local

Descarga `Run_Clean_artifacts.zip`, descomprime y copia (sustituyendo) en
`backend/app/simulators/Azure_model_new_data/resultados/Run_Ultimate/`:

| Del zip | Al repo |
|---|---|
| `checkpoints/best_model.keras` | `.../Run_Ultimate/checkpoints/best_model.keras` |
| `results/scaler.joblib` | `.../Run_Ultimate/results/scaler.joblib` |
| `results/threshold.json` | `.../Run_Ultimate/results/threshold.json` |

El backend adopta los tres automáticamente (el umbral se lee de `threshold.json`).
Después, en local: borra `backend/data/fleet_scores.json`, ejecuta
`python -m app.services.fleet` y `python -m app.ml.evaluate_ensemble`
para la comparativa final ML vs MNF vs ensemble.

**Si `best_model.keras` no carga en local** (Keras de Kaggle más nuevo que el 2.16.2 local):
usa el fallback de pesos — reconstruye la arquitectura con `build_lstm_autoencoder(...)` de
`app/ml/train_clean.py` (mismas unidades que en la celda 4) y haz
`model.load_weights(".../checkpoints/best_model.weights.h5")`, luego `model.save(".../best_model.keras")`
en local para regenerar el artefacto en tu versión.
